Backpropagation from scratch (NumPy)

See exactly how gradients are computed and weights updated, no framework magic

In [8]:
import numpy as np
np.random.seed(42)

# backpropagation from scratch
# network - 2 inputs -> 4 sigmoid (hidden) -> 1 output (sigmoid)
# task learn xor - [0,0] -> 0, [0,1] -> 1, [1,0] -> 1, [1,1] -> 0

X = np.array([[0,0], [0,1], [1,0], [1,1]])
y =  np.array([[0], [1], [1], [0]])

# intitalize weights

W1 = np.random.randn(2,4) * 0.5  # input -> hidden 2 * 4
b1 = np.zeros((1,4))
W2 = np.random.randn(4,1) * 0.5 # hidden - > output (4 * 1)
b2 = np.zeros((1,1))

def sigmoid(x):
  return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

lr = 1.0
print("training backpropagation from scratch")
print('=' * 40)

for epoch in range(5000):
  # forward pass
  z1 = X @ W1 + b1 # linear (4,2) @ (2,4) = (4,4)
  a1 = sigmoid(z1) # activation
  z2 = a1 @ W2 + b2 # linear(4,4) @ (4,1) = (4,1)
  a2 = sigmoid(z2) # output prediction

  # loss calculation
  loss = np.mean((y - a2) ** 2) # MSE LOSS (1/N) factor

  # backward pass
  # note: we use MSE = 1/N (summation of (y - a2)**2). the 1/N scales the gradient but it doesnt change its direction

  # step 1: dL/da2 = 2 (a2 - y) / n
  dL_da2 = 2 * (a2 -y ) / len(y)

  # step 2: da2/dz2 = sigmoid(z2) = a2*(1 - a2)
  da2_dz2 = a2 * (1 - a2)

  # step 3: dL/dz2 = dL/da2 * da2/dz2
  dL_dz2 = dL_da2 * da2_dz2

  # step 4: gradients for W2, b2
  dL_dw2 = a1.T @ dL_dz2
  dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

  # step 5: propagate to hidden layer
  dL_da1 = dL_dz2 * W2.T
  da1_dz1 = a1 * (1 - a1)
  dL_dz1 = dL_da1 * da1_dz1

  # gradients for w1, b1
  dL_dw1 = X.T @ dL_dz1
  dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

  # update weights
  W2 -= lr * dL_dw2
  b2 -= lr * dL_db2
  W1 -= lr * dL_dw1
  b1 -= lr * dL_db1

  if epoch % 1000 == 0:
    print(f"Epoch {epoch:>4d}: Loss = {loss:.6f}")

print(f"\nFinal predictions: {a2.flatten().round(2)}")
print(f"Targets:           {y.flatten()}")
print("\nNetwork learned XOR using backpropagation!")


training backpropagation from scratch
Epoch    0: Loss = 0.255675
Epoch 1000: Loss = 0.203313
Epoch 2000: Loss = 0.005210
Epoch 3000: Loss = 0.001743
Epoch 4000: Loss = 0.001005

Final predictions: [0.03 0.97 0.97 0.03]
Targets:           [0 1 1 0]

Network learned XOR using backpropagation!


PyTorch 2.x autograd + torch.compile

Modern backprop with compiled autograd and gradient inspection


In [10]:
import torch
import torch.nn as nn

# ============================================================
# PYTORCH 2.x AUTOGRAD - Compiled Backpropagation
# XOR problem with modern PyTorch: autograd + torch.compile
# ============================================================


X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(2,8),
    nn.ReLU(),
    nn.Linear(8,1),
    nn.Sigmoid()
)

# compile model for faster training
compiled_model = torch.compile(model)

optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
loss_fn = nn.MSELoss()

print("Training with pytorch")
print('=' * 40)

for epoch in range(2000):
  # forward pass - pytroch records computational graph
  pred = model(X)
  loss = loss_fn(pred, y)

  # backward pass automatic chain rule
  optimizer.zero_grad() # clear older gradients
  loss.backward()
  optimizer.step()

  if epoch % 400 == 0:
    grad_norms = {
        name : p.grad.norm().item()
        for name,p in model.named_parameters()
        if p.grad is not None
    }
    print(f"Epoch {epoch:>4d}: Loss={loss.item():.6f}, "
              f"Grad norms: {grad_norms}")


with torch.inference_mode(): # even faster than no_grad
  preds = model(X)
  print(f"\nPredictions: {preds.flatten().numpy().round(2)}")
  print(f"Targets:     {y.flatten().numpy()}")

# gradient inspection
pred = model(X)
loss = loss_fn(pred, y)
optimizer.zero_grad() # clear gradients first (essential)
loss.backward()

print("Gradient inspection")
for name, param in model.named_parameters():
  g = param.grad
  print(f"  {name:>10s}: shape={list(param.shape)}, "
          f"norm={g.norm():.4f}, "
          f"has_nan={torch.isnan(g).any().item()}")
print("\nAll gradients computed automatically!")


Training with pytorch
Epoch    0: Loss=0.240499, Grad norms: {'0.weight': 0.020057734102010727, '0.bias': 0.01986861415207386, '2.weight': 0.04304676130414009, '2.bias': 0.008345670998096466}
Epoch  400: Loss=0.001173, Grad norms: {'0.weight': 0.0016494629671797156, '0.bias': 0.001478237914852798, '2.weight': 0.0020963267888873816, '2.bias': 1.4988472685217857e-07}
Epoch  800: Loss=0.000285, Grad norms: {'0.weight': 0.0005140341818332672, '0.bias': 0.0004846987721975893, '2.weight': 0.000565230380743742, '2.bias': 2.4690234567970037e-06}
Epoch 1200: Loss=0.000126, Grad norms: {'0.weight': 0.00022343009186442941, '0.bias': 0.00024146981013473123, '2.weight': 0.00026306245126761496, '2.bias': 8.727620297577232e-07}
Epoch 1600: Loss=0.000070, Grad norms: {'0.weight': 0.00013835146091878414, '0.bias': 7.408787496387959e-05, '2.weight': 0.00015030126087367535, '2.bias': 5.408473953139037e-07}

Predictions: [0.01 0.99 0.99 0.01]
Targets:     [0. 1. 1. 0.]
Gradient inspection
    0.weight: sh

Gradient debugging toolkit

Essential techniques for diagnosing and fixing training issues

In [11]:
import torch
import torch.nn as nn

# ============================================================
# GRADIENT DEBUGGING TOOLKIT
# Run these checks when training goes wrong
# ============================================================

model = nn.Sequential(
    nn.Linear(10, 50),
    nn.ReLU(),
    nn.Linear(50, 20),
    nn.ReLU(),
    nn.Linear(20, 1)
)

x = torch.randn(8, 10)
y = torch.randn(8, 1)

# Forward + backward
model.zero_grad()  # Clear gradients first (essential!)
loss = nn.MSELoss()(model(x), y)
loss.backward()

# ========== CHECK 1: Do all parameters have gradients? ==========
print("=== Gradient Existence Check ===")
for name, param in model.named_parameters():
    if param.grad is None:
        print(f"  WARNING {name}: NO GRADIENT!")
    else:
        print(f"  OK {name}: grad_norm = {param.grad.norm():.6f}")

# ========== CHECK 2: Gradient magnitude per layer ==========
print("\n=== Gradient Magnitudes (watch for vanishing/exploding) ===")
for name, param in model.named_parameters():
    if 'weight' in name and param.grad is not None:
        g = param.grad
        print(f"  {name}: mean={g.mean():.6f}, std={g.std():.6f}, "
              f"max={g.abs().max():.6f}")

# ========== CHECK 3: NaN / Inf detection ==========
print("\n=== NaN/Inf Check ===")
has_issues = False
for name, param in model.named_parameters():
    if param.grad is not None:
        if torch.isnan(param.grad).any():
            print(f"  ALERT {name}: Contains NaN!")
            has_issues = True
        if torch.isinf(param.grad).any():
            print(f"  ALERT {name}: Contains Inf!")
            has_issues = True
if not has_issues:
    print("  All gradients are finite")

# ========== CHECK 4: Numerical gradient verification ==========
# Run BEFORE clipping so analytical gradient is unmodified
print("\n=== Numerical Gradient Check (first weight only) ===")
model.zero_grad()
eps = 1e-5
param = list(model.parameters())[0]

# Get analytical gradient (unclipped)
loss = nn.MSELoss()(model(x), y)
loss.backward()
analytical = param.grad[0, 0].item()

# Get numerical gradient
with torch.no_grad():
    orig = param[0, 0].item()
    param[0, 0] = orig + eps
    loss_plus = nn.MSELoss()(model(x), y).item()
    param[0, 0] = orig - eps
    loss_minus = nn.MSELoss()(model(x), y).item()
    param[0, 0] = orig  # restore

numerical = (loss_plus - loss_minus) / (2 * eps)
print(f"  Analytical: {analytical:.6f}")
print(f"  Numerical:  {numerical:.6f}")
print(f"  Relative error: {abs(analytical-numerical)/(abs(analytical)+1e-8):.2e}")
print("  (Should be < 1e-5 for correct implementation)")

# ========== CHECK 5: Gradient clipping ==========
print("\n=== Gradient Clipping Demo ===")
model.zero_grad()
loss = nn.MSELoss()(model(x), y)
loss.backward()
total_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(), max_norm=1.0)
print(f"  Total gradient norm before clip: {total_norm:.4f}")
print(f"  Clipped to max_norm: 1.0")

=== Gradient Existence Check ===
  OK 0.weight: grad_norm = 0.267973
  OK 0.bias: grad_norm = 0.090906
  OK 2.weight: grad_norm = 0.622252
  OK 2.bias: grad_norm = 0.203513
  OK 4.weight: grad_norm = 0.284675
  OK 4.bias: grad_norm = 0.541966

=== Gradient Magnitudes (watch for vanishing/exploding) ===
  0.weight: mean=-0.000346, std=0.011991, max=0.052499
  2.weight: mean=-0.000237, std=0.019686, max=0.147266
  4.weight: mean=0.027728, std=0.058787, max=0.233227

=== NaN/Inf Check ===
  All gradients are finite

=== Numerical Gradient Check (first weight only) ===
  Analytical: 0.001933
  Numerical:  -0.002980
  Relative error: 2.54e+00
  (Should be < 1e-5 for correct implementation)

=== Gradient Clipping Demo ===
  Total gradient norm before clip: 0.9399
  Clipped to max_norm: 1.0
